# PCA-AE validation dashboard

This notebook validates the additive frozen rank-30 PCA plus MLP coefficient autoencoder. PCA-AE has its own configuration, cache, commands, checkpoints, run tree, and shell runner; the direct Conv1D and Conv2D workflow is unchanged. Model selection uses validation data only, and the test split remains untouched until one compressor is frozen.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Run this notebook from within IAFlowCloud.')
    
    PROJECT_ROOT = PROJECT_ROOT.parent

code_path = PROJECT_ROOT / 'Code'
if str(code_path) not in sys.path:
    sys.path.insert(0, str(code_path))

In [ ]:
from iaflow.pca_autoencoder.artifacts import build_pca_autoencoder
from iaflow.comparison import collect_compressor_results
from iaflow.pca_autoencoder.config import load_pca_ae_experiment_template
from iaflow.pca_autoencoder.data import (
    check_pca_ae_cache,
    load_pca_ae_transform,
)

In [ ]:
pca_path = PROJECT_ROOT / 'Data' / 'NLA' / 'PCA'
pca_metrics_path = pca_path / 'PCAValidationMetrics.json'
assert pca_metrics_path.is_file(), f'Missing PCA validation metrics: {pca_metrics_path}'

pca_metrics = json.loads(pca_metrics_path.read_text())
adopted_rank = int(pca_metrics['adopted_precision_rank'])
maximum_relative_error_threshold = float(pca_metrics['maximum_relative_error_threshold'])
adopted_metrics = pca_metrics['ranks'][str(adopted_rank)]
assert adopted_metrics['maximum_relative_error'] < maximum_relative_error_threshold

LATENT_DIMENSIONS = (2, 4, 6, 8, 10)
DEPTHS = ('Depth03', 'Depth04', 'Depth05')
architectures = []
for depth in DEPTHS:
    config_path = PROJECT_ROOT / 'Config' / 'NLA' / 'PCA_AE' / f'{depth}.yaml'
    template = load_pca_ae_experiment_template(config_path)
    transform = load_pca_ae_transform(template)
    assert transform.rank == adopted_rank
    metadata = check_pca_ae_cache(template)
    assert metadata['pca_transform_sha256'] == transform.artifact_sha256
    for latent_dim in LATENT_DIMENSIONS:
        preview_run = Path(template.output.root_directory) / f'Latent{latent_dim:02d}' / 'NotebookPreview'
        config = template.resolve(latent_dim, preview_run)
        model, _surface_norm, _coefficient_norm, _metadata = build_pca_autoencoder(config)
        architectures.append(model.architecture_summary())

print(
    f'Frozen PCA rank {adopted_rank}; '
    f"validation maximum relative error={adopted_metrics['maximum_relative_error']:.4%}."
)
print(f'Validated PCA-AE architecture/latent combinations: {len(architectures)}')
for architecture in architectures:
    print(
        f"{len(architecture['dense_hidden']):d} hidden layers, "
        f"latent={architecture['latent_dim']:02d}, "
        f"parameters={architecture['number_of_parameters']:,}"
    )

pca_ae_results = [
    result
    for result in collect_compressor_results(PROJECT_ROOT)
    if result['model_family'] == 'PCA_AE'
]
print(f'Completed PCA-AE validation runs: {len(pca_ae_results)}')
for result in pca_ae_results:
    print(
        f"{result['depth']:>7s} latent={result['latent_dim']:02d} "
        f"variance={result['variance_recovered']:.8%} "
        f"log10 MSE={result['log10_mse']:.4e}"
    )

print('Run a complete depth sweep with:')
print('caffeinate -i bash Scripts/NLA/Run_PCA_AE.sh Depth03')

Frozen PCA rank 30; validation maximum relative error=4.5019%.
Validated PCA-AE architecture/latent combinations: 15
3 hidden layers, latent=02, parameters=50,944
3 hidden layers, latent=04, parameters=51,010
3 hidden layers, latent=06, parameters=51,076
3 hidden layers, latent=08, parameters=51,142
3 hidden layers, latent=10, parameters=51,208
4 hidden layers, latent=02, parameters=329,472
4 hidden layers, latent=04, parameters=329,538
4 hidden layers, latent=06, parameters=329,604
4 hidden layers, latent=08, parameters=329,670
4 hidden layers, latent=10, parameters=329,736
5 hidden layers, latent=02, parameters=1,132,800
5 hidden layers, latent=04, parameters=1,132,866
5 hidden layers, latent=06, parameters=1,132,932
5 hidden layers, latent=08, parameters=1,132,998
5 hidden layers, latent=10, parameters=1,133,064
Completed PCA-AE validation runs: 0
Run a complete depth sweep with:
caffeinate -i bash Scripts/NLA/Run_PCA_AE.sh Depth03
